# CT organ segmentation — visual proof

This notebook has one job: run the full-quality TotalSegmentator model on one real CT and show the colored organ masks. It uses the official 102-subject sample dataset, verifies the published archive checksum, extracts only the selected CT, and saves everything under Google Drive.

When **Run all** succeeds, the final cell displays the overlay and a volume table. The durable result folder contains:

- `segmentation.nii.gz` — the 117-label mask in the CT's original geometry;
- `preview.png` — CT, colored overlay, and mask side by side;
- `measurements.json` — organ voxel counts and volumes;
- `run_manifest.json` — hashes, geometry checks, model/runtime versions; and
- TotalSegmentator's own statistics and run-report JSON files.

> These are anatomical segmentations for visual quality control. They are not diagnoses and are not yet DICOM SEG files for OHIF.


In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/Hamza09Hamza/doctor_assistant.git"
REPO_REF = "classifier-evaluation-colab"
REPO_DIR = Path("/content/doctor_assistant")

if REPO_DIR.exists():
    status = subprocess.run(
        ["git", "status", "--porcelain"],
        cwd=REPO_DIR,
        capture_output=True,
        text=True,
        check=True,
    )
    if status.stdout.strip():
        raise RuntimeError(
            f"{REPO_DIR} has local changes. Remove that Colab checkout and run again."
        )
    subprocess.run(["git", "fetch", "origin", REPO_REF], cwd=REPO_DIR, check=True)
    subprocess.run(["git", "switch", REPO_REF], cwd=REPO_DIR, check=True)
    subprocess.run(["git", "reset", "--hard", f"origin/{REPO_REF}"], cwd=REPO_DIR, check=True)
else:
    subprocess.run(
        ["git", "clone", "--branch", REPO_REF, REPO_URL, str(REPO_DIR)],
        check=True,
    )

sys.path.insert(0, str(REPO_DIR))
print("Repository ready:", REPO_DIR)


In [ ]:
# Pin the model package and keep NumPy compatible with Colab's Numba installation.
%pip install -q "numpy>=1.26,<2.1" "TotalSegmentator==2.17.0" "matplotlib>=3.8,<4"

import importlib.metadata as metadata

print("TotalSegmentator:", metadata.version("TotalSegmentator"))
print("NumPy:", metadata.version("numpy"))


In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU is attached. In Colab select Runtime → Change runtime type → L4 GPU, then Run all."
    )
print("GPU:", torch.cuda.get_device_name(0))
print("CUDA runtime:", torch.version.cuda)


In [ ]:
from google.colab import drive

drive.mount("/content/drive")
DRIVE_CACHE = Path("/content/drive/MyDrive/doctor_assistant/totalsegmentator_demo")
WORK_DIR = Path("/content/totalsegmentator_work")
WEIGHTS_DIR = DRIVE_CACHE / "weights"
for directory in (DRIVE_CACHE, WORK_DIR, WEIGHTS_DIR):
    directory.mkdir(parents=True, exist_ok=True)
print("Durable results:", DRIVE_CACHE)
print("Fast local work area:", WORK_DIR)
print("Durable model weights:", WEIGHTS_DIR)


In [ ]:
import os

SUBJECT_INDEX = 0
USE_FAST_MODEL = False  # Keep False on an L4 for the full 1.5 mm model.
FORCE_RERUN = False     # Set True only when you intentionally want new inference.

command = [
    sys.executable,
    "-u",
    "scripts/demo_totalsegmentator.py",
    "--drive-cache",
    str(DRIVE_CACHE),
    "--work-dir",
    str(WORK_DIR),
    "--subject-index",
    str(SUBJECT_INDEX),
]
if USE_FAST_MODEL:
    command.append("--fast")
if FORCE_RERUN:
    command.append("--force")

env = {
    **os.environ,
    "PYTHONUNBUFFERED": "1",
    "TOTALSEG_WEIGHTS_PATH": str(WEIGHTS_DIR),
    "MPLCONFIGDIR": "/content/matplotlib-config",
}
subprocess.run(command, cwd=REPO_DIR, env=env, check=True)


In [ ]:
import json
import pandas as pd
from IPython.display import Image, display

latest = json.loads((DRIVE_CACHE / "latest_run.json").read_text())
result_dir = Path(latest["result_dir"])
measurements = json.loads(Path(latest["measurements"]).read_text())
manifest = json.loads(Path(latest["manifest"]).read_text())

display(Image(filename=latest["preview"], width=1200))
display(pd.DataFrame(measurements["structures"]).head(25))

assert manifest["geometry"]["affine_matches_source"] is True
assert manifest["structure_count"] > 0
assert Path(latest["segmentation"]).exists()
print("SUCCESS — segmentation passed geometry and non-empty-mask checks.")
print("Saved result folder:", result_dir)


## What happens after this succeeds

First, inspect the colored overlay. The boundaries should follow the visible organs rather than spilling into unrelated anatomy. This is visual quality control, not clinical validation.

After we confirm this result, the next engineering step is to run the same model on a real de-identified **DICOM CT series** and ask TotalSegmentator 2.17 to emit `dicom_seg`. OHIF already understands that format; we will then add a simple read-only segmentation panel to Clinique Amina.
